# Data Platform Analysis
## Exploratory Data Analysis and Insights

In [ ]:
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Connect to PostgreSQL
conn = psycopg2.connect(
    host='postgres',
    database='datawarehouse',
    user='datauser',
    password='datapass123'
)

print('Connected to PostgreSQL successfully!')

In [ ]:
# Load customers data
customers_df = pd.read_sql_query(
    'SELECT * FROM raw_data.customers',
    conn
)

print(f'Total customers: {len(customers_df)}')
customers_df.head()

In [ ]:
# Load orders data
orders_df = pd.read_sql_query(
    'SELECT * FROM raw_data.orders',
    conn
)

print(f'Total orders: {len(orders_df)}')
print(f'Total revenue: ${orders_df["total_amount"].sum():.2f}')
orders_df.head()

In [ ]:
# Load products data
products_df = pd.read_sql_query(
    'SELECT * FROM raw_data.products',
    conn
)

print(f'Total products: {len(products_df)}')
products_df.head()

In [ ]:
# Sales by date
sales_by_date = orders_df.groupby('order_date').agg({
    'total_amount': 'sum',
    'order_id': 'count'
}).rename(columns={'order_id': 'order_count'})

plt.figure(figsize=(12, 6))
plt.plot(sales_by_date.index, sales_by_date['total_amount'], marker='o', linewidth=2)
plt.title('Daily Sales Trend')
plt.xlabel('Date')
plt.ylabel('Sales Amount ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Top products by revenue
order_items_df = pd.read_sql_query(
    'SELECT oi.*, p.product_name FROM raw_data.order_items oi JOIN raw_data.products p ON oi.product_id = p.product_id',
    conn
)

order_items_df['revenue'] = order_items_df['quantity'] * order_items_df['unit_price']
top_products = order_items_df.groupby('product_name')['revenue'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
top_products.plot(kind='barh')
plt.title('Top 10 Products by Revenue')
plt.xlabel('Revenue ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Customer metrics
customer_metrics = pd.read_sql_query(
    'SELECT * FROM analytics.customer_metrics',
    conn
)

print('Customer Metrics Summary:')
print(customer_metrics.describe())

In [ ]:
# Close connection
conn.close()
print('Connection closed')